### 04. 레이어 합성·통합지수 (P3)

Layer 1(침수취약성) · Layer 3(노출·취약성·대응역량) · CDRI 를 만든 과정과 그 근거 수치를 본다.
계산은 `src/data/layers.py` 와 `src/stages/h06_layers.py` · `h07_cdri.py` 가 한다. 여기서는
**파이프라인이 쓴 것과 같은 함수·같은 산출물**을 불러 중간값을 확인한다. 노트북에서 새로 계산하지 않는다.

수식과 선택 근거는 `docs/METHODOLOGY.md` §3~§5.

In [1]:
import json
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import layers as L
from src.pipeline.graph import Graph
from src.pipeline.report import node_metrics, node_table

GRAPH = Graph.load()
LAYERS = ROOT / "data/processed/layers"
pd.set_option("display.width", 200)

### P3 노드 상태

승인 대기(`awaiting_approval`)는 실패가 아니다. 사람이 확인해야 하류가 열린다.

In [2]:
node_table(*GRAPH.select(phase='P3'))

,이름,게이트,상태,검증,승인,실패 시,node_id
0,데이터 접근신청 현황,H00,pass,code / human,R1,h00_access_requests,h00_access_requests
1,강수량 수집 확인,H00,pass,code / human,-,h00_collect_rainfall,h00_collect_rainfall
2,하천수위 수집 확인,H00,pass,code / human,-,h00_collect_river,h00_collect_river
3,펌프장·하천 목록 수집 확인,H00,pass,code / human,-,h00_collect_small_tables,h00_collect_small_tables
4,SGIS 인구 통계·경계 수집 확인,H00,pass,code / human,-,h00_collect_sgis,h00_collect_sgis
5,토지피복·DEM 수집 확인,H00,pass,code / human,-,h00_collect_geo,h00_collect_geo
6,강수량 원본 검증,H01,pass,code / code / human,R1,h00_collect_rainfall,h01_contract_rainfall
7,하천수위 원본 검증,H01,pass,code / code / human,R1,h00_collect_river,h01_contract_river
8,펌프장·하천 목록 원본 검증,H01,pass,code / code / human,R1,h00_collect_small_tables,h01_contract_small_tables
9,SGIS 인구 통계·경계 원본 검증,H01,pass,code / code / human,R1,h00_collect_sgis,h01_contract_sgis


## 1. Layer 1 — 기후노출 × 도시민감도

국토부 「도시 기후변화 재해취약성분석 지침」 구조다. 두 축을 각각 z-score 합산한 뒤
Jenks 4등급으로 나누고, 두 등급의 합으로 취약성 I~IV 를 준다.

In [3]:
m1 = node_metrics("h06_layer1_flood")
print("격자", f"{m1['n_grid']:,}", "| 순위대상", f"{m1['n_universe']:,}")
pd.DataFrame(m1["class_counts"], index=["전체"]).T.rename(columns={"전체": "격자 수"})

격자 75,400 | 순위대상 10,202


,격자 수
I,4705
II,7897
III,33995
IV,28803


### 1-1. 강수 보간 — 지수 p 를 교차검증으로 고른 근거

IDW 의 거리 지수 $p$ 를 임의로 정하지 않고, 지점을 하나씩 빼고 나머지로 예측해
오차(RMSE)가 가장 작은 값을 골랐다. 변수마다 다른 $p$ 가 뽑힌다.

In [4]:
rows = []
for name, d in m1["idw"]["by_variable"].items():
    row = {"변수": name, "지점수": d["n_stations"], "선택 p": d["power"],
           "지점평균": d["station_mean"], "격자평균": d["grid_mean"],
           "반경밖 대체": d["n_nearest_fallback"]}
    row.update({f"RMSE p={k}": v for k, v in d["loocv_rmse"].items()})
    rows.append(row)
pd.DataFrame(rows)

,변수,지점수,선택 p,지점평균,격자평균,반경밖 대체,RMSE p=1.0,RMSE p=2.0,RMSE p=3.0
0,rain_annual_max_1h,29,1.0,40.803,40.330,514,4.0291,4.0695,4.2107
1,rain_hours_over_30mm,29,1.0,2.176,2.119,514,0.4740,0.4822,0.5012
2,rain_top5_3h,29,3.0,118.480,116.732,514,10.4254,9.4224,9.1525
3,rain_top5_24h,29,2.0,252.568,250.786,514,17.2311,16.7285,17.1753


### 1-2. 민감도 변수의 부호

부호는 계산 **전에** 물리적 근거로 고정했다. 결과를 보고 바꾸지 않는다.
`-1` 은 값이 작을수록 취약하다는 뜻이다(저지대·완경사).

In [5]:
from src.stages.h06_layers import EXPOSURE_SPEC, SENSITIVITY_SPEC

pd.DataFrame([
    {"축": "기후노출" if k in EXPOSURE_SPEC else "도시민감도", "변수": k, "부호": v,
     "z 평균": d["z_mean"], "z 표준편차": d["z_std"]}
    for group in ("exposure", "sensitivity")
    for k, d in m1["composite"][group].items()
    for v in [d["sign"]]
])

,축,변수,부호,z 평균,z 표준편차
0,기후노출,rain_annual_max_1h,1,0.0,1.0
1,기후노출,rain_hours_over_30mm,1,-0.0,1.0
2,기후노출,rain_top5_3h,1,-0.0,1.0
3,기후노출,rain_top5_24h,1,-0.0,1.0
4,도시민감도,rel_elev_m,-1,0.0,1.0
5,도시민감도,slope_deg,-1,0.0,1.0
6,도시민감도,twi,1,-0.0,1.0
7,도시민감도,impervious_frac,1,0.0,1.0
8,도시민감도,river_proximity,1,-0.0,1.0
9,도시민감도,culvert_proximity,1,-0.0,1.0


### 1-3. Jenks 등급 경계와 취약성 매트릭스

Jenks 는 급간 내부 편차제곱합을 최소화하는 경계를 찾는다. 구현이 맞는지는
작은 표본에서 완전탐색 최적해와 대조해 확인했다(`tests/test_layers.py`).

In [6]:
print("노출 등급 경계   :", m1["jenks_breaks"]["exposure"])
print("민감도 등급 경계 :", m1["jenks_breaks"]["sensitivity"])
print()
matrix = pd.DataFrame(
    [[L.ROMAN[L.VULNERABILITY_MATRIX[e + s]] for s in range(1, 5)] for e in range(1, 5)],
    index=[f"노출 {i}등급" for i in range(1, 5)],
    columns=[f"민감도 {j}등급" for j in range(1, 5)],
)
print("취약성 매트릭스 (I 이 가장 취약)")
matrix

노출 등급 경계   : [-2.1485, 0.8003, 3.7842, 8.6334]
민감도 등급 경계 : [-2.0589, 2.244, 7.9783, 26.979]

취약성 매트릭스 (I 이 가장 취약)


,민감도 1등급,민감도 2등급,민감도 3등급,민감도 4등급
노출 1등급,IV,IV,III,III
노출 2등급,IV,III,III,II
노출 3등급,III,III,II,I
노출 4등급,III,II,I,I


### 1-4. 강건성 — 하천 근접·침수예상도를 빼도 순위가 유지되는가

홍재주 외(2015)가 지적한 "하천 인접도에 따른 I등급 과다"와 예상도 의존을 확인한다.
두 변수를 빼고 다시 계산해도 순위 상관이 높으면 특정 변수에 끌려가지 않는다는 뜻이다.

In [7]:
es = m1["exclusion_sensitivity"]
print("제외 변수 :", es["excluded"])
print("순위 상관 :", es["spearman_rho_universe"])
print()
print("검증 상태")
print("  사례지 face-validity :", m1["case_study"])
print("  침수흔적 라벨        :", m1.get("label_available"), "|", m1.get("label_note", ""))

제외 변수 : ['river_proximity', 'culvert_proximity', 'flood_l210_100_depth_m']
순위 상관 : 0.9043

검증 상태
  사례지 face-validity : {'available': True, 'mapping': {'양덕동': ['양덕1동', '양덕2동'], '봉암동': ['봉암동'], '팔용동': ['팔룡동']}, 'unmatched_legal_dong': ['명서동', '사화동'], 'dong_found': ['봉암동', '양덕1동', '양덕2동', '팔룡동'], 'dong_missing': [], 'n_grid': 428, 'high_grade_share': 0.6098, 'base_rate': 0.3747, 'lift': 1.627, 'lift_min': 1.5, 'note': '선행연구 사례지는 독립 성능검증이 아니라 face-validity 점검이다 (하네스 §7)'}
  침수흔적 라벨        : False | 침수흔적 벡터 자료 없음 (그림 파일 0개). 예측 성능을 주장하지 않는다. 2026-09-14 정보공개 회신분 수령 후 이 노드만 재실행한다


## 2. Layer 3 — 노출·취약성·대응역량

노출은 **수**, 취약성은 **비율**로 나눈다. 같은 사람을 두 번 세지 않기 위해서다.

In [8]:
m3 = node_metrics("h06_layer3_vuln")
print("연령 코드북 :", m3["elderly"]["age_codebook"])
print("검증 방법   :", m3["elderly"]["codebook_verified"])
print()
pd.Series({
    "집계구 수": m3["elderly"]["n_aggregation_units"],
    "집계구 조인율(순위대상)": m3["elderly"]["join_rate_universe"],
    "시 전체 65+ 비율": m3["elderly"]["city_elderly_share"],
    "격자 인구가중 65+ 비율": m3["elderly"]["grid_pop_weighted_universe"],
    "격자 단순평균 65+ 비율": m3["elderly"]["grid_unweighted_mean_universe"],
}).to_frame("값")

연령 코드북 : in_age 5세 계급, 65세 이상 = in_age_014 이후
검증 방법   : 노령화지수(to_in_004) 항등식 대조 — src/data/sgis.py 주석



,값
집계구 수,2118.0000
집계구 조인율(순위대상),1.0000
시 전체 65+ 비율,0.1841
격자 인구가중 65+ 비율,0.1908
격자 단순평균 65+ 비율,0.3173


**단순평균(0.32)이 시 전체(0.18)보다 훨씬 높은 이유.** 면적이 넓은 농촌 집계구가 격자를 많이
차지하기 때문이다. 시 전체와 비교할 수 있는 값은 **인구가중 평균**(0.19)이고, 그것이 맞는다.
집계구 하나가 최대 몇 개 격자에 같은 값을 주는지도 함께 기록한다 — 배분 불확실성이다.

In [9]:
print("집계구당 순위대상 격자 수 :", m3["elderly"]["grids_per_aggregation_unit"])
print()
print("대응역량")
cap = m3["capacity"]
pd.Series({
    "대피장소": cap["by_kind"]["shelter"], "방재기관": cap["by_kind"]["facility"],
    "대피소 중앙거리(m)": cap["shelter_dist_median_universe"],
    "방재기관 중앙거리(m)": cap["facility_dist_median_universe"],
    "부족도 평균": cap["capacity_deficit_mean_universe"],
}).to_frame("값")

집계구당 순위대상 격자 수 : {'median': 2.0, 'max': 106, 'note': '집계구 하나가 격자 여러 개에 같은 비율을 준다 — 배분 불확실성 (ANALYSIS_PLAN §4)'}

대응역량


,값
대피장소,444.0000
방재기관,516.0000
대피소 중앙거리(m),360.4000
방재기관 중앙거리(m),348.0000
부족도 평균,0.2361


**취약성 V 에서 빠진 변수.** 자료를 아직 못 구했다. 보고서 한계에 그대로 쓴다.

In [10]:
pd.Series(m3['missing_variables']).to_frame('사유')

,사유
one_person_household_ratio,SGIS 1인가구 미보유 (100m·집계구 모두)
old_building_ratio,GIS건물통합정보 SHP 미확보 (V-World 키 필요)
basement_building_count,건축물대장 지하층수 미확보 (건축HUB 키 필요)


## 3. CDRI 통합

$$\text{CDRI}_i = \prod_{k \in \{H,E,V,D\}} (X'_{ik})^{w_k}, \qquad
X' = 0.05 + 0.95\cdot\text{minmax}(X)$$

하한을 0 이 아니라 0.05 로 두는 이유는, minmax 로 0 이 된 요소 하나 때문에 기하평균 전체가
0 이 되어 순위 정보가 사라지는 것을 막기 위해서다.

In [11]:
m7 = node_metrics("h07_cdri")
print("기본 산식      :", m7["primary_formula"])
print("Layer 2 포함   :", m7["layer2_included_in_primary"], "|", m7["layer2_reason"])
print()
pd.DataFrame(m7["weights"], index=["H", "E", "V", "D"]).T

기본 산식      : geometric_equal
Layer 2 포함   : False | 관로 비공개로 검증 불가 — docs/decisions/001-layer2-design.md



,H,E,V,D
equal,0.2500,0.2500,0.2500,0.2500
entropy,0.0856,0.6385,0.1248,0.1511


**엔트로피 가중은 인구(E)에 64% 를 몰아준다.** 인구 분포의 왜도가 크기 때문이다.
이론 틀(네 요소가 모두 필요조건)과 어긋나므로 기본 산식으로 쓰지 않고 민감도 비교용으로만 쓴다.

### 3-1. 민감도 — 이 연구의 핵심 발견

In [12]:
pd.DataFrame(m7["variants"])

,variant,spearman_rho_vs_primary,top20_overlap,top20_overlap_pct
0,geometric_equal,1.0000,20,1.00
1,additive_equal,0.8781,10,0.50
2,geometric_entropy,0.8477,10,0.50
3,additive_entropy,0.9249,9,0.45


In [13]:
rb = m7["robustness"]
print("중위 순위상관 :", rb["median_spearman_rho"], f"(기준 {rb['min_spearman_required']})")
print("TOP20 최소중첩:", rb["min_top20_overlap"], f"(기준 {rb['min_overlap_required']})")
print("미달 항목     :", rb["unmet"])
print()
print("→ ranking_mode =", m7["ranking_mode"])
print("  ", m7["ranking_mode_note"])
print("  강건 공통집합:", m7["robust_core"])

중위 순위상관 : 0.8781 (기준 0.8)
TOP20 최소중첩: 9 (기준 14)
미달 항목     : ['TOP 20 최소 중첩 9 < 14']

→ ranking_mode = tier
   정밀 순위를 주장하지 않는다. 위험군(tier)과 강건 공통집합으로만 보고한다 — 하네스 H07 분기
  강건 공통집합: {'n': 8, 'note': '가중치·집계형 변형 4개 모두의 TOP 20 에 공통으로 드는 격자'}


전체 순위 경향은 잘 유지되는데($\rho \ge 0.85$) 최상위 20곳은 절반만 겹친다.
상위권은 값 차이가 미세해 산식을 조금만 바꿔도 순서가 뒤집히기 때문이다.

하네스 H07 은 이 경우 **"정밀 순위 대신 위험군 모드로 보고, 재튜닝 금지"** 라고 미리 정했다.
그래서 기준을 낮추지 않고 산출물의 성격을 바꿨다.

### 3-2. 그 밖의 민감도

In [14]:
print("해상도(MAUP) :", m7["maup"])
print()
t = m7["tiering"]
print("등급화 Cohen kappa :", t["cohen_kappa"], "—", t["note"])
pd.DataFrame({"Balica 5등급": t["balica_counts"], "Jenks 5등급": t["jenks_counts"]}).fillna(0).astype(int)

해상도(MAUP) : {'resolution_m': 500, 'n_blocks': 1605, 'spearman_rho_mean_vs_max': 0.8812, 'note': '500m 블록의 평균과 최대 재집계 순위 비교 (Fontecha et al. 2021 해상도 비교 논리)'}

등급화 Cohen kappa : 0.2356 — Moreira et al.(2021) 은 등급화가 가장 민감하다고 지적한다. 두 방식을 병기한다


,Balica 5등급,Jenks 5등급
보통,5831,2551
낮음,2398,3165
높음,1746,1650
매우높음,226,624
매우낮음,1,2212


### 3-3. 주 원인을 백분위로 정한 이유

가법형 구성비의 최댓값을 쓰면 분포가 치우친 요소(인구)가 거의 항상 이겨서 조치가 한쪽으로 쏠린다.
그래서 ANALYSIS_PLAN §5 대로 **구성요소 백분위가 가장 높은 것**을 주 원인으로 쓴다.

In [15]:
print(m7["cdri_summary"]["primary_cause_rule"])
print()
pd.Series(m7["cdri_summary"]["primary_cause_counts"]).to_frame("격자 수")

구성요소 백분위가 가장 높은 것 (ANALYSIS_PLAN §5). 구성비 최댓값은 분포가 치우친 요소가 항상 이겨서 쓰지 않는다



,격자 수
V,2945
E,2778
H,2278
D,2201


### 3-4. 구성요소 상관 — 중복 투입이 없는지 확인

In [16]:
cdri = gpd.read_file(LAYERS / "cdri.gpkg", layer="cdri")
cdri[["H_scaled", "E_scaled", "V_scaled", "D_scaled", "cdri"]].corr().round(3)

,H_scaled,E_scaled,V_scaled,D_scaled,cdri
H_scaled,1.000,0.348,-0.401,-0.095,0.548
E_scaled,0.348,1.000,-0.519,-0.178,0.729
V_scaled,-0.401,-0.519,1.000,0.092,-0.142
D_scaled,-0.095,-0.178,0.092,1.000,0.134
cdri,0.548,0.729,-0.142,0.134,1.000


## 4. 다음 단계

- 승인 3건 (`h03_pump_stations`, `h07_cdri`, `h08_top20_policy`)
- 2026-09-14 침수흔적도 수령 → Layer 1 외적 검증(ROC-AUC)·사후검증·XGBoost challenger
  (`docs/METHODOLOGY.md` §8)